In [1]:
# Cell 1 — Environment setup and data paths
import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# === Project root ===
from pathlib import Path
def _find_proj_root(start=None):
    p = Path(start or __file__).resolve() if start and not isinstance(start, Path) else Path.cwd().resolve()
    for cand in [p] + list(p.parents):
        try:
            if len({d.name for d in cand.iterdir() if d.is_dir()} & {"data","analysis","models","simulation","manifests"}) >= 3:
                return cand
        except: continue
    return Path.cwd().resolve()
ROOT = _find_proj_root()
DATA_RAW = ROOT / "data"
DATA_PROCESSED = ROOT / "analysis"
EXPERIMENTS_OUT = DATA_PROCESSED / "experiments"
EXPERIMENTS_OUT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Data directories:")
print(f" - Raw:        {DATA_RAW.exists()}")
print(f" - Processed:  {DATA_PROCESSED.exists()}")


Project root: /mnt/ssd4tb/Desktop/C-Elegans
Data directories:
 - Raw:        True
 - Processed:  True


In [3]:
# Cell 2 — Load CeNGEN expression + motif data
expr_path = DATA_PROCESSED / "csv" / "gene_expr_CeNGEN_base_mapped.csv"
motif_path = DATA_PROCESSED / "csv" / "motif_participation_profiles.csv"

expr_df = pd.read_csv(expr_path, index_col=0)
motifs_df = pd.read_csv(motif_path, index_col=0)

print(f"Expression matrix: {expr_df.shape} (neurons x genes)")
print(f"Motif matrix: {motifs_df.shape} (neurons x motifs)")

# Align by neuron IDs
common_neurons = list(set(expr_df.index) & set(motifs_df.index))
expr_df = expr_df.loc[common_neurons]
motifs_df = motifs_df.loc[common_neurons]

print(f"Aligned neurons: {len(common_neurons)}")


Expression matrix: (838, 22469) (neurons x genes)
Motif matrix: (1533, 418) (neurons x motifs)
Aligned neurons: 838


In [7]:
# Cell 3 — use WormBase IDs directly for a representative subset + PCA embeddings
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Pick any subset of WBGene IDs that exist in your matrix
# (for the micro-experiment, 8–10 genes is plenty; you can scale up later)
sample_genes = expr_df.columns[:10].tolist()  # first 10 IDs for now

expr_sub = expr_df[sample_genes].fillna(0)

# Run PCA
scaler = StandardScaler()
pca = PCA(n_components=min(10, expr_sub.shape[1]), random_state=42)
expr_embed = pca.fit_transform(scaler.fit_transform(expr_sub))

print(f"✅ Using {expr_sub.shape[1]} WBGene IDs; PCA variance explained = {pca.explained_variance_ratio_.sum():.2f}")
print("Sample gene IDs:", sample_genes)


✅ Using 10 WBGene IDs; PCA variance explained = 1.00
Sample gene IDs: ['WBGene00010957', 'WBGene00010958', 'WBGene00014454', 'WBGene00010959', 'WBGene00010960', 'WBGene00010961', 'WBGene00000829', 'WBGene00010962', 'WBGene00010963', 'WBGene00010964']


In [10]:
# Cell 4 — Build and analyze hypergraph using WormBase IDs (version-safe)
!pip install -q hypernetx

import hypernetx as hnx
import numpy as np
import random

# Threshold for expression
thresh = np.percentile(expr_sub.values.flatten(), 80)

# Construct hyperedges
hyperedges = {}
for gene_id in expr_sub.columns:
    members = expr_sub.index[expr_sub[gene_id] > thresh].tolist()
    if len(members) > 2:
        hyperedges[gene_id] = members

# Include some motifs as extra edges
motif_cols = motifs_df.columns[:5]
for motif in motif_cols:
    neurons = motifs_df.index[motifs_df[motif] > 0].tolist()
    if len(neurons) > 2:
        hyperedges[f"motif_{motif}"] = neurons

# Build hypergraph
H = hnx.Hypergraph(hyperedges)

# Version-safe node/edge count
num_nodes = len(H.nodes)
num_edges = len(H.edges)
print(f"✅ Hypergraph built: {num_nodes} neurons, {num_edges} hyperedges")

# Simple modularity heuristic
def simple_modularity(H):
    edges = list(H.edges)
    overlap_sum = 0
    total_pairs = 0
    for i in range(len(edges)):
        for j in range(i + 1, len(edges)):
            a = set(H.edges[edges[i]])
            b = set(H.edges[edges[j]])
            if len(a | b) == 0:
                continue
            overlap_sum += len(a & b) / len(a | b)
            total_pairs += 1
    return overlap_sum / total_pairs if total_pairs > 0 else 0

mod_real = simple_modularity(H)

# Random baseline
all_nodes = list(H.nodes)
rand_edges = {e: random.sample(all_nodes, k=len(H.edges[e])) for e in H.edges}
H_rand = hnx.Hypergraph(rand_edges)
mod_rand = simple_modularity(H_rand)

print(f"Modularity (rough): real={mod_real:.3f}, random={mod_rand:.3f}")

# Save summary
hyper_out = EXPERIMENTS_OUT / "hypergraph_summary.csv"
pd.DataFrame({
    "num_nodes": [num_nodes],
    "num_edges": [num_edges],
    "mod_real": [mod_real],
    "mod_rand": [mod_rand]
}).to_csv(hyper_out, index=False)

print(f"📁 Saved hypergraph summary → {hyper_out}")


✅ Hypergraph built: 824 neurons, 14 hyperedges
Modularity (rough): real=0.182, random=0.110
📁 Saved hypergraph summary → /mnt/ssd4tb/Desktop/C-Elegans/data/processed/experiments/hypergraph_summary.csv


In [12]:
# Cell 5 — Markov chain transcriptome dynamics using WormBase IDs (fixed index handling)
from scipy.stats import entropy
import numpy as np
import pandas as pd

# Order neurons by pseudo-time (first PCA component)
pc1_order = np.argsort(expr_embed[:, 0])
expr_sorted = expr_sub.iloc[pc1_order]

# Discretize into states
n_states = 5
bins = np.linspace(expr_sorted.values.min(), expr_sorted.values.max(), n_states + 1)

transition_matrices = {}
entropies = {}

for gene_id in expr_sorted.columns:
    states = np.digitize(expr_sorted[gene_id], bins) - 1  # gives 0..n_states
    states = np.clip(states, 0, n_states - 1)  # ensure all indices are valid
    T = np.zeros((n_states, n_states))
    for i in range(len(states) - 1):
        T[states[i], states[i + 1]] += 1
    # normalize rows to probabilities
    T = T / np.maximum(T.sum(axis=1, keepdims=True), 1e-9)
    transition_matrices[gene_id] = T
    entropies[gene_id] = np.nanmean([entropy(row[row > 0]) for row in T if np.any(row > 0)])

entropy_df = pd.DataFrame.from_dict(entropies, orient="index", columns=["entropy"]).sort_values("entropy")

print("🧮 Markov transition entropies (lower = more predictable):")
display(entropy_df.head(10))
print(f"Mean entropy across genes: {entropy_df.entropy.mean():.3f}")

# Save results
markov_out = EXPERIMENTS_OUT / "markov_entropies.csv"
entropy_df.to_csv(markov_out)
print(f"📁 Saved entropy results → {markov_out}")


🧮 Markov transition entropies (lower = more predictable):


,entropy
WBGene00010958,0.000000
WBGene00014454,0.000000
WBGene00010959,0.000000
WBGene00010961,0.000000
WBGene00010963,0.000000
WBGene00010957,0.009234
WBGene00000829,0.009234
WBGene00010960,0.009234
WBGene00010962,0.009234
WBGene00010964,0.009234


Mean entropy across genes: 0.005
📁 Saved entropy results → /mnt/ssd4tb/Desktop/C-Elegans/data/processed/experiments/markov_entropies.csv


In [13]:
# Build pseudo-time from full transcriptome (more stable ordering)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd
from scipy.stats import entropy

# Use ALL genes for ordering
full_scaled = StandardScaler(with_mean=True, with_std=True).fit_transform(expr_df.values)
pca_full = PCA(n_components=3, random_state=42)
full_embed = pca_full.fit_transform(full_scaled)
order = np.argsort(full_embed[:, 0])   # PC1 on full transcriptome
expr_sorted = expr_df.iloc[order]

# Per-gene quantile states (quintiles)
n_states = 5
def gene_states(series, q=5):
    qs = np.quantile(series, np.linspace(0, 1, q+1))
    st = np.digitize(series, qs, right=True) - 1
    return np.clip(st, 0, q-1)

transition_matrices, entropies = {}, {}
for gene_id in expr_sorted.columns[:200]:  # start with 200 genes for speed; scale later
    s = gene_states(expr_sorted[gene_id].values, q=n_states)
    T = np.zeros((n_states, n_states))
    for i in range(len(s)-1):
        T[s[i], s[i+1]] += 1
    T = T / np.maximum(T.sum(axis=1, keepdims=True), 1e-9)
    transition_matrices[gene_id] = T
    entropies[gene_id] = np.nanmean([entropy(row[row > 0]) for row in T if row.sum() > 0])

entropy_df = pd.DataFrame.from_dict(entropies, orient="index", columns=["entropy"]).sort_values("entropy")
print("Markov entropy (lower = more predictable). Mean:", float(entropy_df.entropy.mean()))
display(entropy_df.head(10))


Markov entropy (lower = more predictable). Mean: 0.285920227036749


,entropy
WBGene00021407,0.000000
WBGene00021397,0.000000
WBGene00017578,0.000000
WBGene00044462,0.000000
WBGene00195533,0.000000
WBGene00018929,0.000000
WBGene00044458,0.000000
WBGene00044723,0.000000
WBGene00015999,0.119852
WBGene00001735,0.121591


In [14]:
# Permutation baseline: shuffle the neuron order
perm = np.random.permutation(len(expr_sorted))
expr_perm = expr_df.iloc[perm]

def mean_entropy_on(exprX, n_states=5, take=200):
    ent = []
    for gene_id in exprX.columns[:take]:
        s = gene_states(exprX[gene_id].values, q=n_states)
        T = np.zeros((n_states, n_states))
        for i in range(len(s)-1):
            T[s[i], s[i+1]] += 1
        T = T / np.maximum(T.sum(axis=1, keepdims=True), 1e-9)
        e = np.nanmean([entropy(row[row > 0]) for row in T if row.sum() > 0])
        if not np.isnan(e):
            ent.append(e)
    return float(np.mean(ent)) if ent else np.nan

print("Entropy (pseudo-time):", mean_entropy_on(expr_sorted))
print("Entropy (permuted)   :", mean_entropy_on(expr_perm))


Entropy (pseudo-time): 0.285920227036749
Entropy (permuted)   : 0.9288432570399721


In [15]:
# Pick a more informative gene set (top variance genes)
gene_var = expr_df.var(axis=0).sort_values(ascending=False)
gene_set = gene_var.head(500).index  # start with 500; adjust as needed
expr_top = expr_df.loc[:, gene_set]

# Per-gene top-quantile membership (e.g., top 10%)
q = 0.90
hyperedges = {}
for gene_id in expr_top.columns:
    cutoff = np.quantile(expr_top[gene_id].values, q)
    members = expr_top.index[expr_top[gene_id] >= cutoff].tolist()
    if len(members) >= 3:
        hyperedges[gene_id] = members

# Optionally add motifs as hyperedges (more than first 5)
for motif in motifs_df.columns[:50]:
    neurons = motifs_df.index[motifs_df[motif] > 0].tolist()
    if len(neurons) >= 3:
        hyperedges[f"motif_{motif}"] = neurons

import hypernetx as hnx, random
H = hnx.Hypergraph(hyperedges)
num_nodes, num_edges = len(H.nodes), len(H.edges)
print(f"Hypergraph: {num_nodes} neurons, {num_edges} hyperedges")

def simple_modularity(H):
    edges = list(H.edges)
    overlap_sum, total_pairs = 0, 0
    for i in range(len(edges)):
        for j in range(i+1, len(edges)):
            a, b = set(H.edges[edges[i]]), set(H.edges[edges[j]])
            if a or b:
                overlap_sum += len(a & b) / max(len(a | b), 1)
                total_pairs += 1
    return overlap_sum / total_pairs if total_pairs else 0

mod_real = simple_modularity(H)

# Degree-preserving randomization for baseline (sample size = edge size)
all_nodes = list(H.nodes)
rand_edges = {e: random.sample(all_nodes, k=len(H.edges[e])) for e in H.edges}
H_rand = hnx.Hypergraph(rand_edges)
mod_rand = simple_modularity(H_rand)
print(f"Modularity (rough): real={mod_real:.3f}, random={mod_rand:.3f}")


Hypergraph: 838 neurons, 550 hyperedges
Modularity (rough): real=0.142, random=0.059


In [16]:
# Hyperedge size distribution and node degrees
edge_sizes = [len(set(H.edges[e])) for e in H.edges]
node_deg = [len(H.nodes[n]) for n in H.nodes]
print("Edge size (min/median/max):", min(edge_sizes), np.median(edge_sizes), max(edge_sizes))
print("Node degree (min/median/max):", min(node_deg), np.median(node_deg), max(node_deg))


Edge size (min/median/max): 13 88.0 838
Node degree (min/median/max): 10 49.0 329


In [17]:
# Top/bottom entropy genes
topN = 50
lowE = entropy_df.sort_values("entropy").head(topN)
highE = entropy_df.sort_values("entropy").tail(topN)
lowE.to_csv(EXPERIMENTS_OUT / "markov_low_entropy_top50.csv")
highE.to_csv(EXPERIMENTS_OUT / "markov_high_entropy_top50.csv")
lowE.head(10)


,entropy
WBGene00021407,0.000000
WBGene00021397,0.000000
WBGene00017578,0.000000
WBGene00044462,0.000000
WBGene00195533,0.000000
WBGene00018929,0.000000
WBGene00044458,0.000000
WBGene00044723,0.000000
WBGene00015999,0.119852
WBGene00001735,0.121591


In [22]:
# Cell 6 — Correlate Markov transition predictability with connectome synaptic weights
import numpy as np
import pandas as pd

edges_path = DATA_PROCESSED / "csv" / "merged_edges_with_relatedness.csv"

edges_df = pd.read_csv(edges_path)
print(f"Loaded edges file: {edges_df.shape[0]} rows")

# Standardize column names for safety
edges_df.columns = [c.lower() for c in edges_df.columns]
src_col = [c for c in edges_df.columns if "source" in c][0]
tgt_col = [c for c in edges_df.columns if "target" in c][0]
weight_col = "synapsecount" if "synapsecount" in edges_df.columns else None

# Compute mean outgoing synaptic strength per neuron (Source)
if weight_col:
    neuron_weights = edges_df.groupby(src_col)[weight_col].mean()
else:
    neuron_weights = edges_df.groupby(src_col).size()  # fallback = number of outgoing edges

neuron_weights = neuron_weights.reindex(expr_df.index).fillna(0)

print(f"Computed mean outgoing synaptic weight for {neuron_weights.shape[0]} neurons")

# Correlate each gene's transition pattern with synaptic weights
n_states = 5
results = []

for gene_id in transition_matrices.keys():
    T = transition_matrices[gene_id]
    # approximate “state weight” series by binning neurons according to pseudo-time
    expr_vals = expr_sorted[gene_id].values
    bins = np.quantile(expr_vals, np.linspace(0, 1, n_states + 1))
    states = np.digitize(expr_vals, bins, right=True) - 1
    states = np.clip(states, 0, n_states - 1)
    
    # compute mean outgoing weight per state
    state_means = []
    for st in range(n_states):
        mask = (states == st)
        mean_w = neuron_weights.iloc[mask].mean() if mask.any() else np.nan
        state_means.append(mean_w)
    
    diffs = np.diff(state_means, prepend=state_means[0])
    corr = np.corrcoef(T.flatten(), np.tile(diffs, n_states))[0, 1]
    results.append((gene_id, float(entropy_df.loc[gene_id, "entropy"]), float(corr)))

corr_df = pd.DataFrame(results, columns=["WBGene", "entropy", "weight_corr"])
corr_df.sort_values("weight_corr", ascending=False, inplace=True)

corr_out = EXPERIMENTS_OUT / "markov_weight_corr.csv"
corr_df.to_csv(corr_out, index=False)

print(f"✅ Saved correlation results → {corr_out}")
print("Top correlated genes:")
display(corr_df.head(10))


Loaded edges file: 27727 rows
Computed mean outgoing synaptic weight for 838 neurons
✅ Saved correlation results → /mnt/ssd4tb/Desktop/C-Elegans/data/processed/experiments/markov_weight_corr.csv
Top correlated genes:


,WBGene,entropy,weight_corr
178,WBGene00004047,0.378853,0.030781
186,WBGene00021927,0.414790,0.027569
62,WBGene00019300,0.404291,0.026009
86,WBGene00000913,0.380287,0.021414
29,WBGene00000897,0.375768,0.012948
72,WBGene00004981,0.399953,0.011620
26,WBGene00017928,0.397672,0.011572
180,WBGene00002278,0.400525,0.009975
76,WBGene00020300,0.401227,0.009378
80,WBGene00020297,0.365401,0.009356


In [19]:
import hypernetx as hnx, numpy as np, random

expr_top = expr_df.loc[:, expr_df.var(axis=0).sort_values(ascending=False).head(2000).index]  # focus on top-var genes

def build_hyperedges(exprX, q=0.95, k=None, min_size=5, max_frac=0.3):
    N = exprX.shape[0]
    edges = {}
    for gene in exprX.columns:
        if k is not None:
            idx = np.argsort(exprX[gene].values)[-k:]
            members = exprX.index[idx].tolist()
        else:
            cutoff = np.quantile(exprX[gene].values, q)
            members = exprX.index[exprX[gene] >= cutoff].tolist()
        if len(members) >= min_size and len(members) <= int(max_frac * N):
            edges[gene] = members
    return edges

hyperedges = build_hyperedges(expr_top, q=0.95, k=None, min_size=5, max_frac=0.3)

# Add motif edges with coverage filter
for motif in motifs_df.columns[:100]:
    members = motifs_df.index[motifs_df[motif] > 0].tolist()
    if 5 <= len(members) <= int(0.5 * expr_df.shape[0]):
        hyperedges[f"motif_{motif}"] = members

H = hnx.Hypergraph(hyperedges)
num_nodes, num_edges = len(H.nodes), len(H.edges)

def simple_modularity(H):
    edges = list(H.edges)
    tot, num = 0.0, 0
    for i in range(len(edges)):
        a = set(H.edges[edges[i]])
        for j in range(i+1, len(edges)):
            b = set(H.edges[edges[j]])
            u = len(a|b)
            if u:
                tot += len(a&b)/u
                num += 1
    return tot/num if num else 0

mod_real = simple_modularity(H)
all_nodes = list(H.nodes)
rand_edges = {e: random.sample(all_nodes, k=len(H.edges[e])) for e in H.edges}
H_rand = hnx.Hypergraph(rand_edges)
mod_rand = simple_modularity(H_rand)

print(f"Hypergraph (filtered): nodes={num_nodes}, edges={num_edges}")
print(f"Modularity (rough): real={mod_real:.3f}, random={mod_rand:.3f}")


Hypergraph (filtered): nodes=838, edges=2091
Modularity (rough): real=0.119, random=0.030


In [23]:
# Filter genes: keep only those with enough variance and ≥3 occupied states
import numpy as np, pandas as pd
from scipy.stats import entropy

n_states = 5

def gene_states(series, q=5):
    qs = np.quantile(series, np.linspace(0, 1, q+1))
    st = np.digitize(series, qs, right=True) - 1
    return np.clip(st, 0, q-1)

entropies_filtered = {}
transition_matrices_filtered = {}

expr_sorted = expr_df.iloc[np.argsort(full_embed[:, 0])]  # pseudo-time from full PCA

var_threshold = 1e-6  # adjust if too strict/lenient
for gene_id in expr_sorted.columns[:2000]:  # start with top 2k for speed
    col = expr_sorted[gene_id].values
    if np.var(col) <= var_threshold:
        continue
    s = gene_states(col, q=n_states)
    if len(np.unique(s)) < 3:
        continue  # needs at least 3 distinct states

    T = np.zeros((n_states, n_states))
    for i in range(len(s)-1):
        T[s[i], s[i+1]] += 1
    T = T / np.maximum(T.sum(axis=1, keepdims=True), 1e-9)

    transition_matrices_filtered[gene_id] = T
    entropies_filtered[gene_id] = np.nanmean([entropy(row[row > 0]) for row in T if row.sum() > 0])

entropy_df_filt = pd.DataFrame.from_dict(entropies_filtered, orient="index", columns=["entropy"]).sort_values("entropy")
print("Filtered mean entropy:", float(entropy_df_filt.entropy.mean()))
display(entropy_df_filt.head(10))


Filtered mean entropy: 0.3488591598699455


,entropy
WBGene00016405,0.186581
WBGene00016180,0.190746
WBGene00000679,0.193056
WBGene00021939,0.195516
WBGene00016398,0.196533
WBGene00015311,0.200969
WBGene00006504,0.201040
WBGene00021282,0.207392
WBGene00018073,0.211780
WBGene00004995,0.212232


In [24]:
# Build hyperedges with top-k rule and cap large edges, then project to 2-section graph and run Louvain
import numpy as np, hypernetx as hnx, networkx as nx
from collections import Counter

N = expr_df.shape[0]
k = 40                     # top-k neurons per gene by expression
max_frac = 0.20            # drop edges covering >20% of neurons
min_size = 5

# Choose informative genes (top variance)
top_var_genes = expr_df.var(axis=0).sort_values(ascending=False).head(2000).index
expr_top = expr_df.loc[:, top_var_genes]

edges = {}
for gene in expr_top.columns:
    idx = np.argsort(expr_top[gene].values)[-k:]
    members = expr_top.index[idx].tolist()
    if len(members) >= min_size and len(members) <= int(max_frac * N):
        edges[gene] = members

# add motif edges with coverage filter
for motif in motifs_df.columns[:200]:
    mem = motifs_df.index[motifs_df[motif] > 0].tolist()
    if min_size <= len(mem) <= int(0.5 * N):
        edges[f"motif_{motif}"] = mem

H = hnx.Hypergraph(edges)
print("Hypergraph:", len(H.nodes), "nodes,", len(H.edges), "edges")

# 2-section projection (clique expansion) with Jaccard weights
G = nx.Graph()
for e in H.edges:
    nodes = list(set(H.edges[e]))
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            u, v = nodes[i], nodes[j]
            # accumulate weight by hyperedge co-membership
            G.add_edge(u, v, weight=G.get_edge_data(u, v, {}).get("weight", 0) + 1)

# Convert raw co-membership to Jaccard-ish weight (optional)
# Here we’ll leave counts; Jaccard would need node degrees in hyperedges.

# Louvain
try:
    import community  # python-louvain
    part = community.best_partition(G, weight="weight")
    # modularity on G (graph modularity)
    mod = community.modularity(part, G, weight="weight")
    print(f"Louvain modularity on 2-section: {mod:.3f}")
except Exception as e:
    print("Louvain not available; install python-louvain. Error:", e)


Hypergraph: 837 nodes, 2194 edges
Louvain not available; install python-louvain. Error: No module named 'community'


In [26]:
# Partial Spearman: correlate (expression residuals) vs (weight residuals) after regressing both on PC1
from scipy.stats import spearmanr
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression

edges_df = pd.read_csv(DATA_PROCESSED / "csv" / "merged_edges_with_relatedness.csv")
edges_df.columns = [c.lower() for c in edges_df.columns]
src_col = [c for c in edges_df.columns if "source" in c][0]
weight_col = "synapsecount" if "synapsecount" in edges_df.columns else None

if weight_col:
    neuron_w = edges_df.groupby(src_col)[weight_col].mean().reindex(expr_df.index).fillna(0).values
else:
    neuron_w = edges_df.groupby(src_col).size().reindex(expr_df.index).fillna(0).values

pc1 = full_embed[:, 0]  # pseudo-time covariate

def residualize(y, x):
    x = x.reshape(-1,1)
    m = LinearRegression().fit(x, y)
    return y - m.predict(x)

w_res = residualize(neuron_w, pc1)

rows = []
for gene in expr_df.columns[:2000]:  # limit for speed
    x = expr_df[gene].values
    if np.var(x) < 1e-6:
        continue
    x_res = residualize(x, pc1)
    rho, p = spearmanr(x_res, w_res)
    rows.append((gene, rho, p))

assoc_df = pd.DataFrame(rows, columns=["WBGene", "partial_spearman", "pval"]).sort_values("partial_spearman", ascending=False)
assoc_df.to_csv(EXPERIMENTS_OUT / "expr_weight_partial_spearman.csv", index=False)
assoc_df.head(10)


,WBGene,partial_spearman,pval
1304,WBGene00200751,0.281207,1.074265e-16
1551,WBGene00020288,0.280494,1.291561e-16
870,WBGene00006204,0.280483,1.295078e-16
424,WBGene00021894,0.280397,1.324161e-16
1136,WBGene00016755,0.280044,1.450437e-16
1331,WBGene00235272,0.280044,1.450437e-16
467,WBGene00200607,0.280044,1.450437e-16
375,WBGene00022090,0.280044,1.450437e-16
1665,WBGene00235282,0.279904,1.503485e-16
1343,WBGene00022617,0.279904,1.503485e-16


In [27]:
# For each gene, slope of expression vs PC1; slope of outgoing weight vs PC1; correlate across genes
from sklearn.linear_model import LinearRegression

# slope of weight vs PC1 (single value)
w_slope = LinearRegression().fit(pc1.reshape(-1,1), neuron_w).coef_[0]

rows = []
for gene in expr_df.columns[:2000]:
    x = expr_df[gene].values
    if np.var(x) < 1e-6:
        continue
    slope = LinearRegression().fit(pc1.reshape(-1,1), x).coef_[0]
    rows.append((gene, slope))
gene_slopes = pd.DataFrame(rows, columns=["WBGene", "expr_slope"])

# Correlate gene slopes with the weight slope sign per neuron is another route,
# but simpler: rank genes by |expr_slope| and cross-check with low-entropy gene set.
gene_slopes.sort_values("expr_slope", ascending=False).head(10)


,WBGene,expr_slope
1864,WBGene00001444,30.079355
750,WBGene00004419,22.681497
1409,WBGene00018348,15.828300
1142,WBGene00019055,14.279632
4,WBGene00010960,13.402852
0,WBGene00010957,9.741569
7,WBGene00010962,9.406665
9,WBGene00010964,7.624733
10,WBGene00010965,5.430262
1095,WBGene00004494,4.927627


In [28]:
!pip install -q python-louvain
import community as community_louvain, networkx as nx

# G is your 2-section graph built earlier with 'weight' attributes
part = community_louvain.best_partition(G, weight="weight")
louvain_mod = community_louvain.modularity(part, G, weight="weight")
print(f"Louvain modularity on 2-section: {louvain_mod:.3f}")


Louvain modularity on 2-section: 0.344


In [30]:
from statsmodels.stats.multitest import multipletests
pvals = assoc_df["pval"].values
rej, q, _, _ = multipletests(pvals, method="fdr_bh")
assoc_df["qval"] = q
assoc_df["signif"] = rej
assoc_df.sort_values(["signif","partial_spearman"], ascending=[False,False]).head(20)


,WBGene,partial_spearman,pval,qval,signif
1304,WBGene00200751,0.281207,1.074265e-16,8.470262e-15,True
1551,WBGene00020288,0.280494,1.291561e-16,8.470262e-15,True
870,WBGene00006204,0.280483,1.295078e-16,8.470262e-15,True
424,WBGene00021894,0.280397,1.324161e-16,8.470262e-15,True
1136,WBGene00016755,0.280044,1.450437e-16,8.470262e-15,True
1331,WBGene00235272,0.280044,1.450437e-16,8.470262e-15,True
467,WBGene00200607,0.280044,1.450437e-16,8.470262e-15,True
375,WBGene00022090,0.280044,1.450437e-16,8.470262e-15,True
1665,WBGene00235282,0.279904,1.503485e-16,8.470262e-15,True
1343,WBGene00022617,0.279904,1.503485e-16,8.470262e-15,True


In [32]:
# Map WBGene IDs to names for the top lists
map_path = DATA_PROCESSED/"tsv"/"genome_data_master.tsv"
if not map_path.exists(): map_path = DATA_RAW/"genomes"/"genome_data_master.tsv"
m = pd.read_csv(map_path, sep="\t")
m.columns = [c.lower() for c in m.columns]
wb_col = [c for c in m.columns if "wb" in c or "wormbase" in c][0]
name_col = [c for c in m.columns if "gene" in c and "name" in c][0]
wb2name = dict(zip(m[wb_col], m[name_col]))

# annotate tables
entropy_named = entropy_df_filt.copy()
entropy_named["gene"] = entropy_named.index.map(wb2name).fillna(entropy_named.index)

assoc_named = assoc_df.copy()
assoc_named["gene"] = assoc_named["WBGene"].map(wb2name).fillna(assoc_named["WBGene"])

slopes_named = gene_slopes.copy()
slopes_named["gene"] = slopes_named["WBGene"].map(wb2name).fillna(slopes_named["WBGene"])


IndexError: list index out of range

In [33]:
import pandas as pd
map_path = DATA_PROCESSED / "tsv" / "genome_data_master.tsv"
if not map_path.exists():
    map_path = DATA_RAW / "genomes" / "genome_data_master.tsv"

m = pd.read_csv(map_path, sep="\t", nrows=5)
print("Columns in genome_data_master.tsv:")
print(list(m.columns))
display(m.head())


Columns in genome_data_master.tsv:
['Unnamed: 0', 'pearson_r_strict', 'spearman_r_strict', 'abs_pearson_strict', 'abs_spearman_strict', 'rank_p_strict', 'rank_s_strict', 'rank_product_strict', 'gold_strict', 'gold_umbrella', 'gold_union', 'variance_var_bulk', 'variance_var_aggr', 'dynamic_range_dyn_bulk', 'meta_rank_product', 'meta_score_mean', 'meta_score_median', 'tier']


,Unnamed: 0,pearson_r_strict,spearman_r_strict,abs_pearson_strict,abs_spearman_strict,rank_p_strict,rank_s_strict,rank_product_strict,gold_strict,gold_umbrella,gold_union,variance_var_bulk,variance_var_aggr,dynamic_range_dyn_bulk,meta_rank_product,meta_score_mean,meta_score_median,tier
0,WBGene00000001,-0.153907,0.025017,0.153907,0.025017,8230.0,16281.0,11575.518563,True,True,True,3.974208e+07,0.112091,16281.153907,7688.100679,3.618376e+06,7688.100679,High
1,WBGene00000002,0.165090,0.443188,0.165090,0.443188,7550.0,1568.0,3440.697604,True,True,True,6.775297e+06,0.112091,7549.834910,7539.884362,6.184497e+05,1568.000000,Low
2,WBGene00000003,0.089890,0.087964,0.089890,0.087964,12683.0,12627.0,12654.969024,True,True,True,3.635940e+07,0.112091,12682.912036,8940.190231,3.310817e+06,8940.190231,High
3,WBGene00000004,0.401921,0.387617,0.401921,0.387617,3721.0,2223.0,2876.070757,True,True,True,2.266676e+06,0.112091,3760.612383,6638.990217,2.078089e+05,2223.000000,Very Low
4,WBGene00000005,-0.224984,-0.230942,0.224984,0.230942,5398.0,6058.0,5718.486163,True,True,True,8.420413e+06,0.112091,6093.230942,4947.122912,7.680571e+05,4947.122912,Low
